# Load Modules

In [2]:
import dask_jobqueue
import dask
import dask.distributed as dask_distributed
import os
import numpy as np
import xarray as xr
import pyproj

import functions

# Open dask-cluster

In [3]:
%%time
cluster = dask_jobqueue.SLURMCluster(cores=4,memory='20GB',processes=1,queue='base', walltime='24:00:00',interface='ib0',local_directory='$TMPDIR')
client = dask.distributed.Client(cluster)
cluster.scale(jobs=6)
client

CPU times: user 308 ms, sys: 71.3 ms, total: 379 ms
Wall time: 2.12 s


Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: http://172.18.4.115:8787/status,
Dashboard: http://172.18.4.115:8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://172.18.4.115:38183,Workers: 0
Dashboard: http://172.18.4.115:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


# Define directories

In [4]:
hcs = 300 # horizontal chunk size
path_data = '/gxfs_work/geomar/smomw355/model_data/ocean-only/INALT60.L120-KRS0020/nemo/'
path_mask = path_data + 'suppl/2_INALT60.L120-KRS0020_mesh_mask.nc'
path_data += 'output/2_INALT60.L120-KRS0020_'

path_current = os.getcwd()
path_save = os.path.join(path_current, 'RMS_w_model') 
os.makedirs(path_save, exist_ok=True)

# Define period and region of interest

In [5]:
lon_c=15.7
lat_c=-37.8

pos_west,pos_east,pos_south,pos_north  = lon_c-2,lon_c+2,lat_c-2,lat_c+2

# choose output frequency
frq = '1d'  
# '1d' = daily means (full water column, levels 1 - 120)
# '4h' = 4h means (levels 2 - 41)
# '5d' = a snapshot each fifth day (levels 2 - 41)

# note that there is data from 2010 - 2017, however the first two years are associated with a secondary spin-up and should not be analyzed.
yeara = 2012
yearb = 2017
yearvec = np.arange(yeara,yearb+1)

# Loop over each year and each month to compute and save RMS vertical profiles of w

In [ ]:
ds_mask = xr.open_dataset(path_mask,chunks={'x':hcs,'y':hcs})
month_array=np.arange(1,13)

for yeara in yearvec:

    #Open files
    paths = !ls {path_data + frq + '_' + str(yeara) + '*' + '_grid_T.nc'}
    ds_ssh = xr.open_mfdataset(paths,chunks={'x':hcs,'y':hcs,'depthv':1,'time_counter':-1}).squeeze().rename({'deptht':'z'})
    
    paths = !ls {path_data + frq + '_' + str(yeara) + '*' + '_grid_W.nc'}
    ds_w = xr.open_mfdataset(paths,chunks={'x':hcs,'y':hcs,'depthw':1,'time_counter':-1}).squeeze().rename({'depthw':'z'})
    
    #Resize dataset to the region of interest
    ds_w_region = ds_w.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)
    ds_ssh_region = ds_ssh.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)
    ds_mask_region = ds_mask.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)

    #Resize dataset to the first 1000 m
    mask_1000 = ds_ssh_region.z <= 1000
    ds_mask_region = ds_mask_region.where(mask_1000, drop=True)
    
    mask_1000_w = ds_w_region.z <= 1050
    ds_w_region = ds_w_region.where(mask_1000_w, drop=True)

    #Define a regular metric grid - needed for the eSQG comparison
    dx=2 #regular spacing in km
    lon_grid=ds_ssh_region.nav_lon
    lat_grid=ds_ssh_region.nav_lat
    a = pyproj.Transformer.from_crs(4326,3395).transform(lon_grid,lat_grid) # project WGS84 onto metric grid
    y3 = a[0]
    x3 = a[1]
    x3min = np.nanmin(x3,1)
    y3min = np.nanmin(y3,0)
    Y3min,X3min = np.meshgrid(y3min, x3min)
    x1=(x3-X3min)/1000
    y1=(y3-Y3min)/1000
    x_len = int(np.floor(np.amax(x1))+1)
    y_len = int(np.floor(np.amax(y1))+1)
    x_dim = np.linspace(0, x_len, int(x_len/dx), endpoint=False)
    y_dim = np.linspace(0, y_len, int(y_len/dx), endpoint=False)
    x2, y2 = np.meshgrid(x_dim, y_dim)

    #Loop over each month
    for m in month_array:
        #Select month of interest
        ds_w_region_m = ds_w_region.where(ds_w_region.time_counter.dt.month.isin([m]), drop=True)
        ds_ssh_region_m = ds_ssh_region.where(ds_ssh_region.time_counter.dt.month.isin([m]), drop=True)

        #Interpolate w on the vertical at SSH grid points since the eSQG will produce w estimates at SSH grid points
        w_org = ds_w_region_m.vovecrtz  
        z1=ds_w_region_m.z
        z2=ds_ssh_region_m.z.where(mask_1000, drop=True).rename({"z": "z_new"})
        
        w_gridded_vert = xr.apply_ufunc(interp_to_prof,
                             w_org.chunk(
                {"x": 100, "y": 100, "z":-1,'time_counter':7}), z1, z2,
                             input_core_dims=[["z"], ["z"], ["z_new"]],
                             output_core_dims=[["z_new"]],
                             exclude_dims=set(("z")),
                             vectorize = True,
                             dask="parallelized",
                             output_dtypes=[w_org.dtype]).rename({"z_new": "z"})
        #Mask data
        w_gridded_vert=w_gridded_vert.where(ds_mask_region.tmask==1).squeeze()*60*60*24

        #Interpolate w on the horizontal on the regular 2-km grid
        w_gridded = xr.apply_ufunc(interp_to_grid,
                             w_gridded_vert.chunk(
                {"x": -1, "y": -1, "z":10,'time_counter':7}), x1, y1, x2, y2,
                             input_core_dims=[["y","x"], ["y","x"], ["y","x"], ["new_y","new_x"], ["new_y","new_x"]],
                             output_core_dims=[["new_y","new_x"]],
                             exclude_dims=set(("y","x")),
                             vectorize = True,
                             dask="parallelized",
                             output_dtypes=[w_gridded_vert.dtype]).rename({"new_y": "y", "new_x": "x"})

        #Filter w at 40 km and remove 50 grid points (about 100 km) in each direction. This corresponds to the area
        #where the eSQG reconstruction will be analyzed, to avoid edge effects
        w_filtered = filtering_lanczos(w_gridded, 40).isel(x=slice(50,-50),y=slice(50,-50)).compute()

        #Select w values associated to the tail of the distribution (+/-1 sigma, considering a normal distribution)
        abs_velocity_field = np.abs(w_filtered)
        thresholds = abs_velocity_field.reduce(np.nanpercentile, dim=['x', 'y'], q=68)
        
        #Create a mask for values above the threshold
        mask = abs_velocity_field >= thresholds
        masked_velocity_field_model = w_filtered.where(mask)
        
        #Compute RMS
        RMS_model=(masked_velocity_field_model**2).mean(dim=["x", "y"])**(1/2)

        #Save dataset
        ds_to_save = xr.Dataset(
            data_vars=dict(
                RMS_model=(["time_counter", "z"], RMS_model.values)
            ),
            coords=dict(
                z=("z", z2.values),
                time_counter=(["time_counter"], ds_ssh_region_m.time_counter.values)
            ) 
        )
        
        ds_to_save.to_netcdf(path=path_save+str(yeara)+'_'+str(m)+'.nc')

        print('File saved')

/gxfs_home/geomar/smomw691/miniconda3/envs/py3_std/lib/python3.7/site-packages/xarray/core/indexing.py:1233: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  value = value[(slice(None),) * axis + (subkey,)]
/gxfs_home/geomar/smomw691/miniconda3/envs/py3_std/lib/python3.7/site-packages/xarray/core/indexing.py:1233: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': Tr

File saved


/gxfs_home/geomar/smomw691/miniconda3/envs/py3_std/lib/python3.7/site-packages/xarray/core/indexing.py:1227: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]
/gxfs_home/geomar/smomw691/miniconda3/envs/py3_std/lib/python3.7/site-packages/xarray/core/indexing.py:1227: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[in